# OneVoice V2 ? GIPFormer VI adaptation gate

Notebook n?y benchmark checkpoint GIPFormer PyTorch ch?nh th?c tr?n c?ng manifest v?i runtime ONNX, sau ?? fine-tune RNN-T c? ki?m so?t n?u quality gate th?t b?i. Train/dev ???c t?o t? V1 Vietnamese synthetic audio; test b? lo?i ho?n to?n kh?i training v? ch? ???c benchmark sau khi dev pass. C?n Colab GPU Linux ?? fine-tune; CPU ch? d?ng cho smoke/compatibility.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, subprocess, sys

GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
MANIFEST = Path('/content/drive/MyDrive/onevoice_audio_v1/manifest.jsonl')
WORK_ROOT = DRIVE_ROOT / 'gipformer_vi_adaptation_v1'
MODEL_ROOT = DRIVE_ROOT / 'models/gipformer_pytorch_v1_official'
ICEFALL_ROOT = DRIVE_ROOT / 'model_cache/gipformer/icefall'
REPORT_ROOT = DRIVE_ROOT / 'reports/gipformer_vi_adaptation_v1'
for path in (WORK_ROOT, MODEL_ROOT, ICEFALL_ROOT.parent, REPORT_ROOT): path.mkdir(parents=True, exist_ok=True)
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', GITHUB_REPO, str(REPO)], check=True)
os.chdir(REPO); os.environ['PYTHONUNBUFFERED'] = '1'
if not MANIFEST.is_file(): raise FileNotFoundError(f'Missing VI manifest: {MANIFEST}')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, capture_output=True)
except FileNotFoundError:
    gpu = None
DEVICE = 'cuda' if gpu is not None and gpu.returncode == 0 else 'cpu'
if DEVICE == 'cuda':
    print('GPU:', gpu.stdout.strip())
else:
    print('No GPU detected: CPU mode. Smoke/compatibility runs work; full DEV PyTorch is much slower and fine-tuning must wait for GPU.')


In [ ]:
# Reproducible source + model pins. `model.pt`, `bpe.model`, and `tokens.txt` are copied to Drive.
UPSTREAM_REPO = 'https://github.com/ggroup-ai-lab/gipformer.git'
UPSTREAM_COMMIT = 'c6abf2f244680a3be6ca4cd79c006dd32b8e6322'
UPSTREAM_DIR = Path('/content/gipformer-upstream')
HF_REPO = 'g-group-ai-lab/gipformer-65M-rnnt'
# Global dependencies are used only by the ONNX baseline bridge. The upstream PyTorch stack stays isolated in uv below.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'uv', 'huggingface_hub', 'numpy==2.2.6', 'PyYAML', 'soundfile', 'librosa', 'sherpa-onnx'], check=True)
import sherpa_onnx
print('ONNX baseline runtime:', sherpa_onnx.__version__)
if not (UPSTREAM_DIR / '.git').is_dir(): subprocess.run(['git', 'clone', UPSTREAM_REPO, str(UPSTREAM_DIR)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_DIR), 'fetch', '--depth', '1', 'origin', UPSTREAM_COMMIT], check=True)
subprocess.run(['git', '-C', str(UPSTREAM_DIR), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
from huggingface_hub import HfApi, snapshot_download
HF_REVISION = HfApi().model_info(HF_REPO).sha
snapshot_download(HF_REPO, revision=HF_REVISION, local_dir=str(MODEL_ROOT), allow_patterns=['model.pt', 'bpe.model', 'tokens.txt', 'config.json'])
missing = [p for p in ('model.pt','bpe.model','tokens.txt') if not (MODEL_ROOT / p).is_file()]
if missing: raise FileNotFoundError(f'Missing official PyTorch artifacts: {missing}')
(WORK_ROOT / 'upstream_pin.json').write_text(json.dumps({'source_repo':UPSTREAM_REPO,'source_commit':UPSTREAM_COMMIT,'hf_repo':HF_REPO,'hf_revision':HF_REVISION}, indent=2), encoding='utf-8')
print('Pinned GIPFormer source/model:', UPSTREAM_COMMIT, HF_REVISION)


In [ ]:
# Isolated upstream PyTorch/Icefall stack. Drive FUSE cannot provide uv's file locks,
# so dependency cache stays local; model/report artifacts stay on Drive.
LOCAL_UV_CACHE = Path('/content/.cache/onevoice-uv')
LOCAL_UV_CACHE.mkdir(parents=True, exist_ok=True)
env = {**os.environ, 'UV_CACHE_DIR': str(LOCAL_UV_CACHE)}
command = ['uv', 'sync', '--extra', 'pytorch', '--verbose']
print('> ' + ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=UPSTREAM_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
assert process.stdout is not None
for line in process.stdout: print(line, end='', flush=True)
if process.wait(): raise RuntimeError('Upstream PyTorch/Icefall environment failed; the complete uv log is printed above. Do not continue.')
UPSTREAM_PYTHON = UPSTREAM_DIR / '.venv/bin/python'
probe = 'import json,torch,k2,kaldifeat; print(json.dumps({"torch":torch.__version__,"cuda":torch.cuda.is_available(),"k2":getattr(k2, "__version__", getattr(k2, "__dev_version__", "unknown"))}))'
check = subprocess.run([str(UPSTREAM_PYTHON), '-c', probe], text=True, capture_output=True)
print('GIPFormer environment exit code:', check.returncode)
print('stdout:', check.stdout, end='' if check.stdout.endswith('\n') else '\n')
print('stderr:', check.stderr, end='' if check.stderr.endswith('\n') else '\n')
if check.returncode: raise RuntimeError('Official PyTorch/Icefall imports failed; copy the stderr above. Do not continue.')
status = json.loads(check.stdout)
if DEVICE == 'cuda' and not status.get('cuda'):
    raise RuntimeError(f'Colab exposed a GPU but the upstream PyTorch environment cannot use it: {status}. Restart GPU runtime once, then re-run this cell.')
if not status.get('cuda'):
    DEVICE = 'cpu'
    print('Upstream environment is CPU-only. Continue for smoke/compatibility only; do not fine-tune or run full DEV on CPU.')
else:
    DEVICE = 'cuda'
    print('CUDA-ready upstream environment:', status)


In [ ]:
# Compare current ONNX and official PyTorch on the same DEV/noisy slice. This cell resumes saved PyTorch batches.
import yaml
SMOKE_SAMPLES = 32
RUNTIME_ONNX = DRIVE_ROOT / 'models/gipformer'
if not RUNTIME_ONNX.is_dir(): raise FileNotFoundError(f'Missing runtime ONNX bundle: {RUNTIME_ONNX}')
config = yaml.safe_load((REPO / 'config/config.yaml').read_text(encoding='utf-8'))
config['asr']['gipformer_model_dir'] = str(RUNTIME_ONNX)
CONFIG = WORK_ROOT / 'benchmark_config.yaml'; CONFIG.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding='utf-8')

def run_streaming(command, label):
    print(f'[{label}] > ' + ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED':'1'})
    assert process.stdout is not None
    for line in process.stdout: print(line, end='', flush=True)
    code = process.wait()
    if code: raise RuntimeError(f'{label} failed with exit code {code}; complete traceback is printed above.')

ONNX_REPORT = REPORT_ROOT / 'onnx_dev_noisy_32'
if not (ONNX_REPORT / 'aggregate.json').is_file():
    onnx_command = [sys.executable, 'scripts/benchmark_asr_v2.py', str(MANIFEST), '--direction','vi2en','--split','dev','--audio','noisy','--denoiser','passthrough','--max-samples',str(SMOKE_SAMPLES),'--progress-every','8','--config',str(CONFIG),'--report-dir',str(ONNX_REPORT),'--resume']
    run_streaming(onnx_command, 'GIPFormer ONNX baseline')
else: print('ONNX baseline already complete; skipping.')

PYTORCH_REPORT = REPORT_ROOT / 'pytorch_dev_noisy_32'
command = [sys.executable, 'scripts/benchmark_gipformer_pytorch.py', str(MANIFEST), '--infer-script',str(UPSTREAM_DIR/'infer_pytorch.py'),'--python',str(UPSTREAM_PYTHON),'--model-dir',str(MODEL_ROOT),'--icefall-dir',str(ICEFALL_ROOT),'--device',DEVICE,'--split','dev','--audio','noisy','--max-samples',str(SMOKE_SAMPLES),'--batch-size','8','--resume','--baseline-aggregate',str(ONNX_REPORT/'aggregate.json'),'--max-regression-pp','1.0','--report-dir',str(PYTORCH_REPORT)]
run_streaming(command, 'GIPFormer PyTorch compatibility')


In [ ]:
result = json.loads((PYTORCH_REPORT / 'aggregate.json').read_text(encoding='utf-8'))
print(json.dumps(result, ensure_ascii=False, indent=2))
if not result['compatibility_gate']['passed']: raise RuntimeError('Gate is not approved; keep the current ONNX runtime.')
decision = {'status':'PYTORCH_CHECKPOINT_COMPATIBLE','training_started':False,'next_required_work':'Run full dev baseline, then obtain or build a reviewed Icefall Vietnamese construction training recipe.','report':str(PYTORCH_REPORT/'aggregate.json')}
(WORK_ROOT/'compatibility_decision.json').write_text(json.dumps(decision, ensure_ascii=False, indent=2), encoding='utf-8')
print('PASS: official checkpoint matches the ONNX baseline within 1 percentage point.')


## Full development-set baseline

Chạy sau smoke pass. Cell này đo toàn bộ `dev` clean và noisy, không đọc `test`. Nó có thể mất nhiều thời gian; mỗi batch đã hoàn tất được lưu vào Drive và chạy lại sẽ resume. Đây là baseline để chọn candidate fine-tune ở bước sau, không phải fine-tune.


In [ ]:
# Full DEV benchmark: every physical clean WAV exactly once, plus every noisy WAV.
# The V1 logical manifest repeats each clean WAV for its noisy augmentations.
# Never modify the source manifest; write a reproducible canonical subset instead.
FULL_DEV_BATCH_SIZE = 64
CLEAN_MANIFEST = WORK_ROOT / 'vi_dev_clean_unique_manifest.jsonl'
run_streaming([
    sys.executable, 'scripts/build_asr_manifest_subset.py', str(MANIFEST),
    '--output', str(CLEAN_MANIFEST), '--split', 'dev', '--language', 'vi',
    '--audio', 'clean', '--dedupe-audio',
], 'Build canonical VI dev/clean manifest')

benchmark_inputs = {
    'clean_unique': (CLEAN_MANIFEST, 'clean'),
    'noisy': (MANIFEST, 'noisy'),
}
full_reports = {}
for label, (benchmark_manifest, audio_kind) in benchmark_inputs.items():
    onnx_dir = REPORT_ROOT / f'onnx_dev_full_{label}'
    pytorch_dir = REPORT_ROOT / f'pytorch_dev_full_{label}'
    if not (onnx_dir / 'aggregate.json').is_file():
        run_streaming([
            sys.executable, 'scripts/benchmark_asr_v2.py', str(benchmark_manifest),
            '--direction', 'vi2en', '--split', 'dev', '--audio', audio_kind,
            '--denoiser', 'passthrough', '--max-samples', '0', '--progress-every', '64',
            '--config', str(CONFIG), '--report-dir', str(onnx_dir), '--resume',
        ], f'GIPFormer ONNX full dev/{label}')
    else:
        print(f'ONNX full dev/{label} already complete; skipping.')
    if not (pytorch_dir / 'aggregate.json').is_file():
        run_streaming([
            sys.executable, 'scripts/benchmark_gipformer_pytorch.py', str(benchmark_manifest),
            '--infer-script', str(UPSTREAM_DIR / 'infer_pytorch.py'),
            '--python', str(UPSTREAM_PYTHON), '--model-dir', str(MODEL_ROOT),
            '--icefall-dir', str(ICEFALL_ROOT), '--device', DEVICE, '--split', 'dev',
            '--audio', audio_kind, '--max-samples', '0', '--batch-size', str(FULL_DEV_BATCH_SIZE),
            '--resume', '--baseline-aggregate', str(onnx_dir / 'aggregate.json'),
            '--max-regression-pp', '1.0', '--report-dir', str(pytorch_dir),
        ], f'GIPFormer PyTorch full dev/{label}')
    else:
        print(f'PyTorch full dev/{label} already complete; skipping.')
    full_reports[label] = json.loads((pytorch_dir / 'aggregate.json').read_text(encoding='utf-8'))

(WORK_ROOT / 'full_dev_compatibility.json').write_text(
    json.dumps(full_reports, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(full_reports, ensure_ascii=False, indent=2))
print('Full dev compatibility complete. The old duplicated-clean report is retained only as a failed historical artifact; do not use it.')
print('Keep test untouched; the next task is the reviewed train/dev adaptation recipe.')


## Fine-tune VI construction RNN-T

Baseline full dev ?? x?c nh?n checkpoint PyTorch/ONNX t??ng th?ch nh?ng critical-term recall ch?a ??t 95%. Ba cell d??i ??y chu?n b? train/dev, fine-tune t? checkpoint ch?nh th?c v?i checkpoint-resume theo epoch, r?i benchmark dev. Kh?ng s?a manifest g?c v? kh?ng d?ng test ?? ch?n model.


In [ ]:
# Prepare VI train/dev only. This is deterministic and never emits test rows.
if DEVICE != 'cuda':
    raise RuntimeError('GIPFormer fine-tuning requires a GPU runtime. CPU is for benchmark smoke only.')

PREPARED_ROOT = DRIVE_ROOT / 'datasets/gipformer_vi_construction_v1'
run_streaming([
    sys.executable, 'scripts/prepare_gipformer_finetune_data.py', str(MANIFEST),
    '--output-dir', str(PREPARED_ROOT),
], 'Prepare GIPFormer VI train/dev')

prepared_info = json.loads((PREPARED_ROOT / 'dataset_manifest.json').read_text(encoding='utf-8'))
print(json.dumps(prepared_info, ensure_ascii=False, indent=2))
if prepared_info['test_split_included']:
    raise RuntimeError('Safety stop: test split appeared in prepared training data.')


In [ ]:
# Fine-tune the official 65M RNN-T checkpoint. Re-running resumes after the last completed epoch.
FINE_TUNE_DIR = DRIVE_ROOT / 'models/gipformer_vi_construction_v1'
command = [
    str(UPSTREAM_PYTHON), 'scripts/finetune_gipformer_rnnt.py',
    '--train', str(PREPARED_ROOT / 'train.jsonl'),
    '--dev', str(PREPARED_ROOT / 'dev.jsonl'),
    '--model-dir', str(MODEL_ROOT),
    '--icefall-dir', str(ICEFALL_ROOT),
    '--output', str(FINE_TUNE_DIR),
    '--epochs', '5',
    '--batch-size', '4',
    '--learning-rate', '1e-5',
    '--save-every-steps', '500',
    '--resume', 'auto',
    '--device', 'cuda',
]
run_streaming(command, 'Fine-tune GIPFormer VI RNN-T')
print((FINE_TUNE_DIR / 'training_summary.json').read_text(encoding='utf-8'))


In [ ]:
# Select only on development data. Test remains untouched unless this dev quality gate passes.
if not (FINE_TUNE_DIR / 'best' / 'model.pt').is_file():
    raise FileNotFoundError('Best adapted checkpoint is missing; fine-tuning did not complete.')

adapted_dev_reports = {}
for label, (benchmark_manifest, audio_kind) in benchmark_inputs.items():
    baseline_dir = REPORT_ROOT / f'onnx_dev_full_{label}'
    report_dir = REPORT_ROOT / f'pytorch_finetuned_dev_{label}'
    if not (report_dir / 'aggregate.json').is_file():
        run_streaming([
            sys.executable, 'scripts/benchmark_gipformer_pytorch.py', str(benchmark_manifest),
            '--infer-script', str(UPSTREAM_DIR / 'infer_pytorch.py'),
            '--python', str(UPSTREAM_PYTHON), '--model-dir', str(FINE_TUNE_DIR / 'best'),
            '--icefall-dir', str(ICEFALL_ROOT), '--device', 'cuda', '--split', 'dev',
            '--audio', audio_kind, '--max-samples', '0', '--batch-size', '64', '--resume',
            '--baseline-aggregate', str(baseline_dir / 'aggregate.json'),
            '--max-regression-pp', '1.0', '--report-dir', str(report_dir),
        ], f'Adapted GIPFormer dev/{label}')
    adapted_dev_reports[label] = json.loads((report_dir / 'aggregate.json').read_text(encoding='utf-8'))

(DEVELOPMENT_GATE := FINE_TUNE_DIR / 'development_gate.json').write_text(
    json.dumps(adapted_dev_reports, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(adapted_dev_reports, ensure_ascii=False, indent=2))
noisy_gate = adapted_dev_reports['noisy']['critical_term_recall'] >= 0.95
clean_gate = adapted_dev_reports['clean_unique']['critical_term_recall'] >= 0.95
print(f'Dev quality gate: clean={clean_gate}, noisy={noisy_gate}')
if not (clean_gate and noisy_gate):
    print('Do not touch test or promote this checkpoint. Inspect dev errors and revise the training recipe.')


In [ ]:
# Fixed test is a final report only, never a training or model-selection input.
if not (clean_gate and noisy_gate):
    raise RuntimeError('Dev quality gate failed; test remains intentionally untouched.')

TEST_CLEAN_MANIFEST = WORK_ROOT / 'vi_test_clean_unique_manifest.jsonl'
run_streaming([
    sys.executable, 'scripts/build_asr_manifest_subset.py', str(MANIFEST),
    '--output', str(TEST_CLEAN_MANIFEST), '--split', 'test', '--language', 'vi',
    '--audio', 'clean', '--dedupe-audio',
], 'Build canonical VI test/clean manifest')

final_test_reports = {}
for label, (benchmark_manifest, audio_kind) in {
    'clean_unique': (TEST_CLEAN_MANIFEST, 'clean'),
    'noisy': (MANIFEST, 'noisy'),
}.items():
    report_dir = REPORT_ROOT / f'pytorch_finetuned_test_{label}'
    if not (report_dir / 'aggregate.json').is_file():
        run_streaming([
            sys.executable, 'scripts/benchmark_gipformer_pytorch.py', str(benchmark_manifest),
            '--infer-script', str(UPSTREAM_DIR / 'infer_pytorch.py'),
            '--python', str(UPSTREAM_PYTHON), '--model-dir', str(FINE_TUNE_DIR / 'best'),
            '--icefall-dir', str(ICEFALL_ROOT), '--device', 'cuda', '--split', 'test',
            '--audio', audio_kind, '--max-samples', '0', '--batch-size', '64', '--resume',
            '--report-dir', str(report_dir),
        ], f'Adapted GIPFormer test/{label}')
    final_test_reports[label] = json.loads((report_dir / 'aggregate.json').read_text(encoding='utf-8'))
print(json.dumps(final_test_reports, ensure_ascii=False, indent=2))
